In [2]:
# ============================================================
# PUBMED mHEALTH DATASET EXTRACTION
# Fresh proposed-system corpus
#
# Years: 2015–2025
# Language: English
# Abstract required
# Reviews / Meta-analyses / Editorials etc. excluded
# Search is restricted to mHealth-specific title/abstract terms
# ============================================================

%pip install -q biopython pandas tqdm

from Bio import Entrez
import pandas as pd
import time
from tqdm import tqdm

# ============================================================
# 1. NCBI SETTINGS
# ============================================================

# Put your own email here
Entrez.email = "YOUR_EMAIL@example.com"

# API key is OPTIONAL.
# You can leave this commented out.
# Entrez.api_key = "YOUR_NCBI_API_KEY"

# NCBI recommends identifying E-utilities requests
# with a valid email and keeping request rates controlled.
# We deliberately use delays below.
# ============================================================


# ============================================================
# 2. SEARCH SETTINGS
# ============================================================

START_YEAR = 2015
END_YEAR = 2025

# IMPORTANT:
# We are NOT using just "mHealth" because PubMed's
# automatic term mapping can make the search very broad.
#
# We are restricting the main concepts to title/abstract.
#
MH_TERMS = """
(
    "mHealth"[tiab]
    OR "m-health"[tiab]
    OR "mobile health"[tiab]
    OR "mobile healthcare"[tiab]
    OR "mobile health app"[tiab]
    OR "mobile health apps"[tiab]
    OR "mobile health application"[tiab]
    OR "mobile health applications"[tiab]
    OR "mobile health intervention"[tiab]
    OR "mobile health interventions"[tiab]
    OR "mobile health technology"[tiab]
    OR "mobile health technologies"[tiab]
)
"""

# Publication filters
PUBLICATION_FILTERS = """
AND english[lang]
AND hasabstract
NOT review[pt]
NOT systematic review[pt]
NOT meta-analysis[pt]
NOT editorial[pt]
NOT comment[pt]
NOT letter[pt]
NOT news[pt]
NOT preprint[pt]
NOT retracted publication[pt]
NOT retraction of publication[pt]
"""

BASE_QUERY = f"""
{MH_TERMS}
{PUBLICATION_FILTERS}
"""


# ============================================================
# 3. PUBMED SEARCH FUNCTION
# ============================================================

def search_pubmed_year(year):

    query = f"""
    (
        {BASE_QUERY}
    )
    AND {year}/01/01:{year}/12/31[dp]
    """

    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmode="xml",
        retmax=10000,
        sort="pub date"
    )

    record = Entrez.read(handle)
    handle.close()

    return [str(x) for x in record["IdList"]]


# ============================================================
# 4. SEARCH YEAR BY YEAR
# ============================================================

all_pmids = []

print("=" * 70)
print("SEARCHING PUBMED")
print("=" * 70)

for year in range(START_YEAR, END_YEAR + 1):

    pmids = search_pubmed_year(year)

    print(f"{year}: {len(pmids):,} records")

    all_pmids.extend(pmids)

    # Safe delay
    time.sleep(0.5)


# Remove duplicate PMIDs
all_pmids = list(dict.fromkeys(all_pmids))

print("\n" + "=" * 70)
print(f"TOTAL UNIQUE PMIDs: {len(all_pmids):,}")
print("=" * 70)


# ============================================================
# 5. FETCH RECORDS IN BATCHES
# ============================================================

def fetch_pubmed_records(pmids, batch_size=300):

    articles = []

    for start in tqdm(
        range(0, len(pmids), batch_size),
        desc="Downloading PubMed records"
    ):

        batch = pmids[start:start + batch_size]

        handle = Entrez.efetch(
            db="pubmed",
            id=batch,
            retmode="xml"
        )

        records = Entrez.read(handle)
        handle.close()

        articles.extend(records["PubmedArticle"])

        # Safe delay between requests
        time.sleep(0.5)

    return articles


articles = fetch_pubmed_records(
    all_pmids,
    batch_size=300
)

print(
    f"\nDownloaded articles: {len(articles):,}"
)


# ============================================================
# 6. PARSE PUBMED ARTICLE
# ============================================================

def parse_pubmed_article(article):

    medline = article["MedlineCitation"]
    article_data = medline["Article"]

    # --------------------------------------------------------
    # PMID
    # --------------------------------------------------------

    pmid = str(
        medline["PMID"]
    )

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title = str(
        article_data.get(
            "ArticleTitle",
            ""
        )
    )

    # --------------------------------------------------------
    # ABSTRACT
    # --------------------------------------------------------

    abstract = ""

    if "Abstract" in article_data:

        abstract_parts = []

        for item in article_data[
            "Abstract"
        ]["AbstractText"]:

            text = str(item)

            # Preserve structured abstract labels
            try:
                label = item.attributes.get(
                    "Label"
                )

                if label:
                    text = f"{label}: {text}"

            except Exception:
                pass

            abstract_parts.append(text)

        abstract = " ".join(
            abstract_parts
        )

    # --------------------------------------------------------
    # YEAR
    # --------------------------------------------------------

    year = ""

    try:

        pub_date = (
            article_data
            ["Journal"]
            ["JournalIssue"]
            ["PubDate"]
        )

        if "Year" in pub_date:

            year = str(
                pub_date["Year"]
            )

        elif "MedlineDate" in pub_date:

            year = str(
                pub_date["MedlineDate"]
            )[:4]

    except Exception:

        pass

    # --------------------------------------------------------
    # JOURNAL
    # --------------------------------------------------------

    journal = str(
        article_data
        ["Journal"]
        .get(
            "Title",
            ""
        )
    )

    # --------------------------------------------------------
    # AUTHORS
    # --------------------------------------------------------

    authors = []

    try:

        if "AuthorList" in article_data:

            for author in article_data[
                "AuthorList"
            ]:

                if "CollectiveName" in author:

                    name = str(
                        author[
                            "CollectiveName"
                        ]
                    )

                else:

                    first = str(
                        author.get(
                            "ForeName",
                            ""
                        )
                    )

                    last = str(
                        author.get(
                            "LastName",
                            ""
                        )
                    )

                    name = (
                        f"{first} {last}"
                        .strip()
                    )

                if name:

                    authors.append(
                        name
                    )

    except Exception:

        pass

    authors_text = "; ".join(
        authors
    )

    # --------------------------------------------------------
    # DOI
    # --------------------------------------------------------

    doi = ""

    try:

        article_id_list = (
            article
            .get(
                "PubmedData",
                {}
            )
            .get(
                "ArticleIdList",
                []
            )
        )

        for article_id in article_id_list:

            try:

                id_type = (
                    article_id
                    .attributes
                    .get("IdType")
                )

                if id_type == "doi":

                    doi = str(
                        article_id
                    )

                    break

            except Exception:

                pass

    except Exception:

        pass

    # --------------------------------------------------------
    # MeSH TERMS
    # --------------------------------------------------------

    mesh_terms = []

    try:

        if "MeshHeadingList" in medline:

            for mesh in medline[
                "MeshHeadingList"
            ]:

                mesh_terms.append(
                    str(
                        mesh[
                            "DescriptorName"
                        ]
                    )
                )

    except Exception:

        pass

    mesh_text = "; ".join(
        mesh_terms
    )

    # --------------------------------------------------------
    # KEYWORDS
    # --------------------------------------------------------

    keywords = []

    try:

        if "KeywordList" in medline:

            for keyword_group in medline[
                "KeywordList"
            ]:

                for keyword in keyword_group:

                    keywords.append(
                        str(keyword)
                    )

    except Exception:

        pass

    keyword_text = "; ".join(
        keywords
    )

    # --------------------------------------------------------
    # PUBLICATION TYPES
    # --------------------------------------------------------

    publication_types = []

    try:

        if "PublicationTypeList" in article_data:

            for pt in article_data[
                "PublicationTypeList"
            ]:

                publication_types.append(
                    str(pt)
                )

    except Exception:

        pass

    publication_types_text = "; ".join(
        publication_types
    )

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return {

        "pmid": pmid,

        "title": title,

        "abstract": abstract,

        "year": year,

        "journal": journal,

        "authors": authors_text,

        "doi": doi,

        "mesh_terms": mesh_text,

        "keywords": keyword_text,

        "publication_types":
            publication_types_text
    }


# ============================================================
# 7. PARSE ALL RECORDS
# ============================================================

rows = []

print(
    "\nParsing PubMed records..."
)

for article in tqdm(
    articles,
    desc="Parsing records"
):

    try:

        rows.append(
            parse_pubmed_article(
                article
            )
        )

    except Exception as e:

        print(
            "Parsing error:",
            e
        )


df = pd.DataFrame(
    rows
)


# ============================================================
# 8. DATA CLEANING
# ============================================================

# Remove duplicate PMIDs
df = df.drop_duplicates(
    subset=["pmid"]
)

# Convert year
df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

# Keep only 2015–2025
df = df[
    (df["year"] >= START_YEAR) &
    (df["year"] <= END_YEAR)
]

# Require title
df = df[
    df["title"]
    .fillna("")
    .str.strip()
    != ""
]

# Require abstract
df = df[
    df["abstract"]
    .fillna("")
    .str.strip()
    != ""
]

# Sort
df = df.sort_values(
    ["year", "pmid"]
).reset_index(
    drop=True
)


# ============================================================
# 9. ADD COMBINED TEXT COLUMN
# ============================================================

df["document"] = (
    df["title"].fillna("")
    + ". "
    + df["abstract"].fillna("")
)


# ============================================================
# 10. SAVE DATASET
# ============================================================

OUTPUT_FILE = (
    "pubmed_mhealth_2015_2025_proposed_raw.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("FINAL PUBMED DATASET")
print("=" * 70)

print(
    f"Total records: {len(df):,}"
)

print(
    f"Missing PMID: "
    f"{df['pmid'].isna().sum()}"
)

print(
    f"Missing title: "
    f"{df['title'].isna().sum()}"
)

print(
    f"Missing abstract: "
    f"{df['abstract'].isna().sum()}"
)

print(
    "Duplicate PMID:",
    df["pmid"].duplicated().sum()
)

print("\nRecords by year:")

print(
    df["year"]
    .value_counts()
    .sort_index()
)

print("\nPublication types:")

print(
    df["publication_types"]
    .value_counts()
    .head(20)
)

print("\nColumns:")

print(
    df.columns.tolist()
)

print("\nSaved file:")

print(
    OUTPUT_FILE
)

print("=" * 70)

Note: you may need to restart the kernel to use updated packages.
SEARCHING PUBMED
2015: 508 records
2016: 724 records
2017: 913 records
2018: 1,133 records
2019: 1,369 records
2020: 1,737 records
2021: 1,858 records
2022: 2,082 records
2023: 1,860 records
2024: 1,863 records
2025: 2,290 records

TOTAL UNIQUE PMIDs: 15,020


IncompleteRead: IncompleteRead(1994 bytes read)

In [3]:
# ============================================================
# SAFE / RESUMABLE PUBMED DOWNLOADER
# Uses the existing all_pmids list
# ============================================================

import os
import json
import time
from pathlib import Path
from tqdm import tqdm
from Bio import Entrez

# ------------------------------------------------------------
# NCBI identity
# ------------------------------------------------------------
Entrez.email = "bhuvanavijayam19@gmail.com"

# ------------------------------------------------------------
# Check that the search results still exist
# ------------------------------------------------------------
print("Total PMIDs available:", len(all_pmids))

if len(all_pmids) == 0:
    raise ValueError(
        "all_pmids is empty. Re-run the PubMed search cell first."
    )

# ------------------------------------------------------------
# Checkpoint directory
# ------------------------------------------------------------
CHECKPOINT_DIR = Path("pubmed_download_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Smaller batch = safer
BATCH_SIZE = 100

# Number of retries for a failed request
MAX_RETRIES = 5


# ------------------------------------------------------------
# Fetch one batch with retries
# ------------------------------------------------------------
def fetch_one_batch(pmids, batch_number):

    checkpoint_file = (
        CHECKPOINT_DIR /
        f"batch_{batch_number:04d}.json"
    )

    # Already downloaded?
    if checkpoint_file.exists():

        print(
            f"Batch {batch_number}: already saved → skipping"
        )

        with open(
            checkpoint_file,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    # Try multiple times
    for attempt in range(1, MAX_RETRIES + 1):

        try:

            print(
                f"Batch {batch_number}: "
                f"attempt {attempt}/{MAX_RETRIES}"
            )

            handle = Entrez.efetch(
                db="pubmed",
                id=pmids,
                retmode="xml"
            )

            records = Entrez.read(handle)

            handle.close()

            articles = records["PubmedArticle"]

            # ------------------------------------------------
            # Convert XML records into JSON-safe dictionaries
            # ------------------------------------------------
            batch_rows = []

            for article in articles:

                try:

                    row = parse_pubmed_article(article)

                    batch_rows.append(row)

                except Exception as e:

                    print(
                        f"  Parsing error in batch "
                        f"{batch_number}: {e}"
                    )

            # ------------------------------------------------
            # Save checkpoint immediately
            # ------------------------------------------------
            with open(
                checkpoint_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    batch_rows,
                    f,
                    ensure_ascii=False
                )

            print(
                f"  ✅ Batch {batch_number} saved "
                f"({len(batch_rows)} records)"
            )

            return batch_rows

        except Exception as e:

            print(
                f"  ⚠️ Batch {batch_number} failed: {e}"
            )

            if attempt < MAX_RETRIES:

                wait_time = min(
                    5 * (2 ** (attempt - 1)),
                    60
                )

                print(
                    f"  Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                print(
                    f"  ❌ Batch {batch_number} "
                    f"failed after {MAX_RETRIES} attempts."
                )

                return []


# ------------------------------------------------------------
# Download all batches
# ------------------------------------------------------------
all_rows = []

num_batches = (
    len(all_pmids) + BATCH_SIZE - 1
) // BATCH_SIZE

print("\n" + "=" * 70)
print("RESUMABLE PUBMED DOWNLOAD")
print("=" * 70)

print(
    f"Total PMIDs : {len(all_pmids):,}"
)

print(
    f"Batch size  : {BATCH_SIZE}"
)

print(
    f"Total batches: {num_batches}"
)

print("=" * 70)


for batch_idx in tqdm(
    range(num_batches),
    desc="Downloading PubMed records"
):

    start = batch_idx * BATCH_SIZE
    end = min(
        start + BATCH_SIZE,
        len(all_pmids)
    )

    batch_pmids = all_pmids[start:end]

    batch_rows = fetch_one_batch(
        batch_pmids,
        batch_idx + 1
    )

    all_rows.extend(batch_rows)

    # Stay comfortably below NCBI's no-key rate guidance
    time.sleep(1)


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------
df = pd.DataFrame(all_rows)

print("\n" + "=" * 70)
print("DOWNLOAD COMPLETE")
print("=" * 70)

print(
    f"Downloaded records: {len(df):,}"
)

print(
    f"Unique PMIDs: {df['pmid'].nunique():,}"
)

Total PMIDs available: 15020

RESUMABLE PUBMED DOWNLOAD
Total PMIDs : 15,020
Batch size  : 100
Total batches: 151


Batch 1: attempt 1/5
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' is not defined
  Parsing error in batch 1: name 'parse_pubmed_article' 

Batch 2: attempt 1/5
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' is not defined
  Parsing error in batch 2: name 'parse_pubmed_article' 

Batch 3: attempt 1/5
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' is not defined
  Parsing error in batch 3: name 'parse_pubmed_article' 

Batch 4: attempt 1/5
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' is not defined
  Parsing error in batch 4: name 'parse_pubmed_article' 

...

Batch 17: attempt 1/5
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_pubmed_article' is not defined
  Parsing error in batch 17: name 'parse_p

Batch 18: attempt 1/5
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_pubmed_article' is not defined
  Parsing error in batch 18: name 'parse_p

Batch 19: attempt 1/5
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_pubmed_article' is not defined
  Parsing error in batch 19: name 'parse_p

Batch 20: attempt 1/5


KeyboardInterrupt: 

In [4]:
# ============================================================
# PUBMED ARTICLE PARSER
# Run this cell BEFORE the downloader cell
# ============================================================

def parse_pubmed_article(article):

    medline = article["MedlineCitation"]
    article_data = medline["Article"]

    # PMID
    pmid = str(medline["PMID"])

    # TITLE
    title = str(
        article_data.get("ArticleTitle", "")
    )

    # ABSTRACT
    abstract = ""

    if "Abstract" in article_data:

        abstract_parts = []

        for item in article_data["Abstract"]["AbstractText"]:

            text = str(item)

            try:
                label = item.attributes.get("Label")

                if label:
                    text = f"{label}: {text}"

            except Exception:
                pass

            abstract_parts.append(text)

        abstract = " ".join(abstract_parts)

    # YEAR
    year = ""

    try:

        pub_date = (
            article_data["Journal"]
            ["JournalIssue"]
            ["PubDate"]
        )

        if "Year" in pub_date:

            year = str(pub_date["Year"])

        elif "MedlineDate" in pub_date:

            year = str(
                pub_date["MedlineDate"]
            )[:4]

    except Exception:
        pass

    # JOURNAL
    journal = str(
        article_data["Journal"].get(
            "Title", ""
        )
    )

    # AUTHORS
    authors = []

    try:

        if "AuthorList" in article_data:

            for author in article_data["AuthorList"]:

                if "CollectiveName" in author:

                    name = str(
                        author["CollectiveName"]
                    )

                else:

                    first = str(
                        author.get(
                            "ForeName", ""
                        )
                    )

                    last = str(
                        author.get(
                            "LastName", ""
                        )
                    )

                    name = f"{first} {last}".strip()

                if name:
                    authors.append(name)

    except Exception:
        pass

    authors_text = "; ".join(authors)

    # DOI
    doi = ""

    try:

        article_id_list = (
            article
            .get("PubmedData", {})
            .get("ArticleIdList", [])
        )

        for article_id in article_id_list:

            try:

                if (
                    article_id.attributes.get("IdType")
                    == "doi"
                ):

                    doi = str(article_id)
                    break

            except Exception:
                pass

    except Exception:
        pass

    # MeSH TERMS
    mesh_terms = []

    try:

        if "MeshHeadingList" in medline:

            for mesh in medline["MeshHeadingList"]:

                mesh_terms.append(
                    str(mesh["DescriptorName"])
                )

    except Exception:
        pass

    mesh_text = "; ".join(mesh_terms)

    # KEYWORDS
    keywords = []

    try:

        if "KeywordList" in medline:

            for keyword_group in medline["KeywordList"]:

                for keyword in keyword_group:

                    keywords.append(
                        str(keyword)
                    )

    except Exception:
        pass

    keyword_text = "; ".join(keywords)

    # PUBLICATION TYPES
    publication_types = []

    try:

        if "PublicationTypeList" in article_data:

            for pt in article_data["PublicationTypeList"]:

                publication_types.append(
                    str(pt)
                )

    except Exception:
        pass

    publication_types_text = "; ".join(
        publication_types
    )

    return {
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "year": year,
        "journal": journal,
        "authors": authors_text,
        "doi": doi,
        "mesh_terms": mesh_text,
        "keywords": keyword_text,
        "publication_types": publication_types_text
    }

print("✅ parse_pubmed_article() is ready")

✅ parse_pubmed_article() is ready


In [ ]:
# ============================================================
# PUBMED mHEALTH DATASET EXTRACTION
# Fresh Proposed-System Dataset
#
# Period: 2015-2025
# Language: English
# Abstract required
#
# EXCLUDES:
#   Review
#   Systematic Review
#   Meta-Analysis
#   Editorial
#   Comment
#   Letter
#   News
#   Preprint
#   Retractions
#
# NO NCBI API KEY REQUIRED
#
# Output:
#   pubmed_mhealth_2015_2025_proposed_final.csv
# ============================================================

%pip install -q biopython pandas tqdm

# ============================================================
# IMPORTS
# ============================================================

import os
import io
import json
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from Bio import Entrez


# ============================================================
# NCBI SETTINGS
# ============================================================

# IMPORTANT: change this to YOUR email
Entrez.email = "bhuvanavijayam19@gmail.com"

# No API key required.
# Do NOT add Entrez.api_key unless you have one.

# Identify the software making the requests
Entrez.tool = "mHealth_Research_Project"

# Built-in Biopython retry settings
Entrez.max_tries = 5
Entrez.sleep_between_tries = 10


# ============================================================
# SEARCH SETTINGS
# ============================================================

START_YEAR = 2015
END_YEAR = 2025

# Focused mHealth search.
# We deliberately do NOT include broad terms such as
# smartphone* or wearable* by themselves because they produce
# huge numbers of unrelated healthcare papers.

MH_TERMS = """
(
    "mHealth"[tiab]
    OR "m-health"[tiab]
    OR "mobile health"[tiab]
    OR "mobile healthcare"[tiab]
    OR "mobile health app"[tiab]
    OR "mobile health apps"[tiab]
    OR "mobile health application"[tiab]
    OR "mobile health applications"[tiab]
    OR "mobile health intervention"[tiab]
    OR "mobile health interventions"[tiab]
    OR "mobile health technology"[tiab]
    OR "mobile health technologies"[tiab]
)
"""

PUBLICATION_FILTERS = """
AND english[lang]
AND hasabstract
NOT Review[pt]
NOT Systematic Review[pt]
NOT Meta-Analysis[pt]
NOT Editorial[pt]
NOT Comment[pt]
NOT Letter[pt]
NOT News[pt]
NOT Preprint[pt]
NOT Retracted Publication[pt]
NOT Retraction of Publication[pt]
"""

BASE_QUERY = f"""
{MH_TERMS}
{PUBLICATION_FILTERS}
"""


# ============================================================
# CHECKPOINT SETTINGS
# ============================================================

CHECKPOINT_DIR = Path(
    "pubmed_mhealth_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    exist_ok=True
)

BATCH_SIZE = 100

MAX_BATCH_RETRIES = 5

# Wait between NCBI requests.
# 0.5 sec = comfortably below 3 requests/sec.
REQUEST_DELAY = 0.5


# ============================================================
# STEP 1
# SEARCH PUBMED YEAR BY YEAR
# ============================================================

def search_pubmed_year(year):

    query = f"""
    (
        {BASE_QUERY}
    )
    AND {year}/01/01:{year}/12/31[dp]
    """

    for attempt in range(1, MAX_BATCH_RETRIES + 1):

        try:

            handle = Entrez.esearch(
                db="pubmed",
                term=query,
                retmode="xml",
                retmax=10000,
                sort="pub date"
            )

            result = Entrez.read(handle)

            handle.close()

            pmids = [
                str(x)
                for x in result["IdList"]
            ]

            count = int(
                result["Count"]
            )

            print(
                f"{year}: {count:,} matching records"
            )

            # Safety warning
            if count > 10000:

                print(
                    f"WARNING: {year} has "
                    f"{count:,} records, exceeding "
                    f"the 10,000 retmax used here."
                )

            time.sleep(
                REQUEST_DELAY
            )

            return pmids

        except Exception as e:

            print(
                f"Search failed for {year} "
                f"(attempt {attempt}/{MAX_BATCH_RETRIES}):"
            )

            print(e)

            if attempt < MAX_BATCH_RETRIES:

                wait = min(
                    5 * attempt,
                    60
                )

                print(
                    f"Waiting {wait} seconds..."
                )

                time.sleep(wait)

            else:

                raise


# ============================================================
# RUN YEAR-BY-YEAR SEARCH
# ============================================================

print("=" * 70)
print("PUBMED mHEALTH SEARCH")
print("=" * 70)

all_pmids = []

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    pmids = search_pubmed_year(
        year
    )

    all_pmids.extend(
        pmids
    )


# Remove duplicates
all_pmids = list(
    dict.fromkeys(
        all_pmids
    )
)

print("\n" + "=" * 70)
print(
    f"TOTAL UNIQUE PMIDs: {len(all_pmids):,}"
)
print("=" * 70)


# ============================================================
# SAVE PMID LIST
# ============================================================

pmid_file = (
    "pubmed_mhealth_2015_2025_pmids.txt"
)

with open(
    pmid_file,
    "w",
    encoding="utf-8"
) as f:

    for pmid in all_pmids:
        f.write(
            pmid + "\n"
        )

print(
    f"PMID list saved to: {pmid_file}"
)


# ============================================================
# STEP 2
# PARSER
# ============================================================

def parse_pubmed_article(article):

    medline = article[
        "MedlineCitation"
    ]

    article_data = medline[
        "Article"
    ]

    # --------------------------------------------------------
    # PMID
    # --------------------------------------------------------

    pmid = str(
        medline["PMID"]
    )

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title = str(
        article_data.get(
            "ArticleTitle",
            ""
        )
    )

    # --------------------------------------------------------
    # ABSTRACT
    # --------------------------------------------------------

    abstract_parts = []

    if "Abstract" in article_data:

        for item in article_data[
            "Abstract"
        ]["AbstractText"]:

            text = str(item)

            # Preserve structured abstract label
            try:

                label = item.attributes.get(
                    "Label"
                )

                if label:
                    text = (
                        f"{label}: {text}"
                    )

            except Exception:

                pass

            abstract_parts.append(
                text
            )

    abstract = " ".join(
        abstract_parts
    ).strip()

    # --------------------------------------------------------
    # PUBLICATION DATE / YEAR
    # --------------------------------------------------------

    year = ""

    try:

        pub_date = (
            article_data
            ["Journal"]
            ["JournalIssue"]
            ["PubDate"]
        )

        if "Year" in pub_date:

            year = str(
                pub_date["Year"]
            )

        elif "MedlineDate" in pub_date:

            year = str(
                pub_date["MedlineDate"]
            )[:4]

    except Exception:

        pass

    # --------------------------------------------------------
    # JOURNAL
    # --------------------------------------------------------

    journal = str(
        article_data
        ["Journal"]
        .get(
            "Title",
            ""
        )
    )

    # --------------------------------------------------------
    # AUTHORS
    # --------------------------------------------------------

    author_list = []

    try:

        if "AuthorList" in article_data:

            for author in article_data[
                "AuthorList"
            ]:

                if "CollectiveName" in author:

                    name = str(
                        author[
                            "CollectiveName"
                        ]
                    )

                else:

                    first = str(
                        author.get(
                            "ForeName",
                            ""
                        )
                    )

                    last = str(
                        author.get(
                            "LastName",
                            ""
                        )
                    )

                    name = (
                        f"{first} {last}"
                        .strip()
                    )

                if name:
                    author_list.append(
                        name
                    )

    except Exception:

        pass

    authors = "; ".join(
        author_list
    )

    # --------------------------------------------------------
    # DOI
    # --------------------------------------------------------

    doi = ""

    try:

        article_ids = (
            article
            .get(
                "PubmedData",
                {}
            )
            .get(
                "ArticleIdList",
                []
            )
        )

        for article_id in article_ids:

            try:

                id_type = (
                    article_id
                    .attributes
                    .get(
                        "IdType"
                    )
                )

                if id_type == "doi":

                    doi = str(
                        article_id
                    )

                    break

            except Exception:

                continue

    except Exception:

        pass

    # --------------------------------------------------------
    # MeSH TERMS
    # --------------------------------------------------------

    mesh_terms = []

    try:

        if "MeshHeadingList" in medline:

            for mesh in medline[
                "MeshHeadingList"
            ]:

                descriptor = str(
                    mesh[
                        "DescriptorName"
                    ]
                )

                mesh_terms.append(
                    descriptor
                )

    except Exception:

        pass

    mesh_text = "; ".join(
        mesh_terms
    )

    # --------------------------------------------------------
    # KEYWORDS
    # --------------------------------------------------------

    keywords = []

    try:

        if "KeywordList" in medline:

            for keyword_group in medline[
                "KeywordList"
            ]:

                for keyword in keyword_group:

                    keywords.append(
                        str(keyword)
                    )

    except Exception:

        pass

    keyword_text = "; ".join(
        keywords
    )

    # --------------------------------------------------------
    # PUBLICATION TYPES
    # --------------------------------------------------------

    publication_types = []

    try:

        if "PublicationTypeList" in article_data:

            for pt in article_data[
                "PublicationTypeList"
            ]:

                publication_types.append(
                    str(pt)
                )

    except Exception:

        pass

    publication_type_text = "; ".join(
        publication_types
    )

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return {

        "pmid": pmid,

        "title": title,

        "abstract": abstract,

        "year": year,

        "journal": journal,

        "authors": authors,

        "doi": doi,

        "mesh_terms": mesh_text,

        "keywords": keyword_text,

        "publication_types":
            publication_type_text
    }


# ============================================================
# STEP 3
# FETCH ONE BATCH
# ============================================================

def fetch_batch(
    batch_pmids,
    batch_number
):

    checkpoint_file = (
        CHECKPOINT_DIR
        / f"batch_{batch_number:04d}.json"
    )

    # --------------------------------------------------------
    # If already downloaded, use checkpoint
    # --------------------------------------------------------

    if checkpoint_file.exists():

        try:

            with open(
                checkpoint_file,
                "r",
                encoding="utf-8"
            ) as f:

                rows = json.load(
                    f
                )

            print(
                f"Batch {batch_number}: "
                f"checkpoint found → "
                f"{len(rows)} records"
            )

            return rows

        except Exception:

            print(
                f"Batch {batch_number}: "
                f"checkpoint is corrupted → "
                f"redownloading"
            )

            checkpoint_file.unlink(
                missing_ok=True
            )

    # --------------------------------------------------------
    # Download with retries
    # --------------------------------------------------------

    for attempt in range(
        1,
        MAX_BATCH_RETRIES + 1
    ):

        try:

            print(
                f"Batch {batch_number}: "
                f"attempt {attempt}/"
                f"{MAX_BATCH_RETRIES}"
            )

            handle = Entrez.efetch(
                db="pubmed",
                id=batch_pmids,
                retmode="xml"
            )

            # Read the COMPLETE response first
            raw_xml = handle.read()

            handle.close()

            # Basic corruption check
            if not raw_xml:

                raise ValueError(
                    "Empty response received"
                )

            if (
                b"<PubmedArticleSet"
                not in raw_xml
            ):

                raise ValueError(
                    "Incomplete/invalid XML response"
                )

            # Parse only after entire response
            xml_handle = io.BytesIO(
                raw_xml
            )

            records = Entrez.read(
                xml_handle
            )

            articles = records[
                "PubmedArticle"
            ]

            # Parse articles
            rows = []

            for article in articles:

                try:

                    rows.append(
                        parse_pubmed_article(
                            article
                        )
                    )

                except Exception as e:

                    print(
                        f"  Article parsing error: {e}"
                    )

            # ------------------------------------------------
            # Save checkpoint immediately
            # ------------------------------------------------

            with open(
                checkpoint_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    rows,
                    f,
                    ensure_ascii=False
                )

            print(
                f"  ✅ Batch {batch_number} saved "
                f"({len(rows)} records)"
            )

            time.sleep(
                REQUEST_DELAY
            )

            return rows

        except Exception as e:

            print(
                f"  ⚠️ Batch {batch_number} failed:"
            )

            print(
                f"  {type(e).__name__}: {e}"
            )

            if attempt < MAX_BATCH_RETRIES:

                wait = min(
                    5 * (2 ** (attempt - 1)),
                    60
                )

                print(
                    f"  Waiting {wait} seconds..."
                )

                time.sleep(
                    wait
                )

            else:

                print(
                    f"  ❌ Batch {batch_number} "
                    f"failed after "
                    f"{MAX_BATCH_RETRIES} attempts."
                )

                raise


# ============================================================
# STEP 4
# DOWNLOAD ALL RECORDS
# ============================================================

num_batches = (
    len(all_pmids)
    + BATCH_SIZE
    - 1
) // BATCH_SIZE

print("\n" + "=" * 70)
print("DOWNLOADING PUBMED RECORDS")
print("=" * 70)

print(
    f"Total PMIDs : {len(all_pmids):,}"
)

print(
    f"Batch size  : {BATCH_SIZE}"
)

print(
    f"Total batches: {num_batches}"
)

print(
    f"Checkpoint folder: {CHECKPOINT_DIR}"
)

print("=" * 70)


all_rows = []

for batch_idx in tqdm(
    range(num_batches),
    desc="Downloading PubMed records"
):

    start = (
        batch_idx
        * BATCH_SIZE
    )

    end = min(
        start
        + BATCH_SIZE,
        len(all_pmids)
    )

    batch_pmids = all_pmids[
        start:end
    ]

    rows = fetch_batch(
        batch_pmids,
        batch_idx + 1
    )

    all_rows.extend(
        rows
    )


# ============================================================
# STEP 5
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    all_rows
)


# ============================================================
# STEP 6
# CLEAN DATA
# ============================================================

if df.empty:

    raise ValueError(
        "No records were downloaded."
    )


# Remove duplicate PMID
df = df.drop_duplicates(
    subset=["pmid"]
)


# Convert year
df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)


# Keep only 2015–2025
df = df[
    (df["year"] >= START_YEAR)
    &
    (df["year"] <= END_YEAR)
]


# Require title
df = df[
    df["title"]
    .fillna("")
    .str.strip()
    != ""
]


# Require abstract
df = df[
    df["abstract"]
    .fillna("")
    .str.strip()
    != ""
]


# ============================================================
# STEP 7
# CREATE COMBINED DOCUMENT
# ============================================================

df["document"] = (
    df["title"]
    .fillna("")
    .str.strip()
    + ". "
    +
    df["abstract"]
    .fillna("")
    .str.strip()
)


# ============================================================
# STEP 8
# FINAL SORT
# ============================================================

df = df.sort_values(
    ["year", "pmid"]
).reset_index(
    drop=True
)


# ============================================================
# STEP 9
# FINAL QUALITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET QUALITY CHECK")
print("=" * 70)

print(
    f"Total records: {len(df):,}"
)

print(
    f"Unique PMIDs: "
    f"{df['pmid'].nunique():,}"
)

print(
    f"Missing PMID: "
    f"{df['pmid'].isna().sum()}"
)

print(
    f"Missing title: "
    f"{df['title'].isna().sum()}"
)

print(
    f"Missing abstract: "
    f"{df['abstract'].isna().sum()}"
)

print(
    f"Duplicate PMID: "
    f"{df['pmid'].duplicated().sum()}"
)

print(
    f"Year outside range: "
    f"{(
        (df['year'] < START_YEAR)
        |
        (df['year'] > END_YEAR)
    ).sum()}"
)


# ============================================================
# STEP 10
# YEAR DISTRIBUTION
# ============================================================

print("\nRecords by year:")

print(
    df["year"]
    .value_counts()
    .sort_index()
)


# ============================================================
# STEP 11
# PUBLICATION TYPE CHECK
# ============================================================

print("\nTop publication types:")

print(
    df["publication_types"]
    .value_counts()
    .head(20)
)


# ============================================================
# STEP 12
# SAVE FINAL DATASET
# ============================================================

OUTPUT_FILE = (
    "pubmed_mhealth_2015_2025_proposed_final.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 70)
print("✅ DOWNLOAD COMPLETE")
print("=" * 70)

print(
    f"Final records: {len(df):,}"
)

print(
    f"Final file: {OUTPUT_FILE}"
)

print(
    f"Checkpoint folder: "
    f"{CHECKPOINT_DIR}"
)

print("=" * 70)

Note: you may need to restart the kernel to use updated packages.
PUBMED mHEALTH SEARCH
2015: 508 matching records
2016: 724 matching records
2017: 913 matching records
2018: 1,133 matching records
2019: 1,369 matching records
2020: 1,737 matching records
2021: 1,858 matching records
2022: 2,082 matching records
2023: 1,860 matching records
2024: 1,863 matching records
2025: 2,290 matching records

TOTAL UNIQUE PMIDs: 15,020
PMID list saved to: pubmed_mhealth_2015_2025_pmids.txt

DOWNLOADING PUBMED RECORDS
Total PMIDs : 15,020
Batch size  : 100
Total batches: 151
Checkpoint folder: pubmed_mhealth_checkpoints


Batch 1: checkpoint found → 100 records
Batch 2: checkpoint found → 100 records
Batch 3: checkpoint found → 100 records
Batch 4: checkpoint found → 100 records
Batch 5: attempt 1/5
  ✅ Batch 5 saved (100 records)


Batch 6: attempt 1/5
  ✅ Batch 6 saved (100 records)


Batch 7: attempt 1/5
  ✅ Batch 7 saved (100 records)


Batch 8: attempt 1/5
  ✅ Batch 8 saved (100 records)


...

Batch 141: attempt 1/5
  ✅ Batch 141 saved (100 records)


Batch 142: attempt 1/5
  ✅ Batch 142 saved (100 records)


Batch 143: attempt 1/5
  ✅ Batch 143 saved (100 records)


Batch 144: attempt 1/5
  ✅ Batch 144 saved (100 records)


Batch 145: attempt 1/5
